In [ ]:
import pandas as pd
import numpy as np
import scipy.io
import matplotlib.pyplot as plt

In [ ]:
file_name= 'B0005.mat'
mat = scipy.io.loadmat(file_name)

In [ ]:
battery_name = file_name.split('.')[0]
cycles = mat[battery_name]['cycle'][0][0][0]

extracted_data = []
cycle_number = 1

In [ ]:
for cycle in cycles:
    cycle_type= cycle['type'][0]

    if cycle_type == 'discharge':
        data = cycle['data'][0][0]

        voltage_measured= data['Voltage_measured'][0]
        temperature_measured = data['Temperature_measured'][0]

        capacity = data['Capacity'][0][0]

        extracted_data.append({ 
            'Cycle_Number': cycle_number,
            'Capacity': capacity,
            'Voltage_measured': voltage_measured,
            'Temperature_measured': temperature_measured
        })

        cycle_number += 1
        

In [ ]:
df = pd.DataFrame(extracted_data)

print(f"Dataframe shape: {df.shape}")
print(df[['Cycle_Number', 'Capacity', 'Voltage_measured', 'Temperature_measured']].head())

In [ ]:
# Assuming your previous dataframe from Step 2 is named 'df'

engineered_features = []

for index, row in df.iterrows():
    volts = row['Voltage_measured']
    temps= row['Temperature_measured']

    engineered_features.append({
        'Cycle_Number': row['Cycle_Number'],
        'Max_Temperature': np.max(temps),
        'Average_Temperature': np.mean(temps),
        'Min_Voltage': np.min(volts),
        'Voltage_Drop': np.max(volts) - np.min(volts),
        'Capacity': row['Capacity']
    })

ml_df = pd.DataFrame(engineered_features)
print(ml_df.head())
         

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

In [ ]:
X = ml_df[['Cycle_Number', 'Max_Temperature', 'Average_Temperature','Min_Voltage','Voltage_Drop']]
y = ml_df['Capacity']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)


In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

In [ ]:
predictions = lr_model.predict(X_test)

In [ ]:
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print(f"Mean Absolute Error (MAE): {mae:4f} Ah")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f} Ah")

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
test_cycles = X_test['Cycle_Number']

plt.figure(figsize=(12,6))

In [ ]:
plt.plot(test_cycles, y_test, label='Actual Capacity (Ground Truth)', color='blue', linewidth=2)
plt.plot(test_cycles, predictions, label='Predicted Capacity (Linear Regression)', color='red', linestyle='--', linewidth='2')
plt.axhline(y=1.4, color='black', linestyle=':', linewidth=2, label='End of Life Threshold (1.4 Ah)')

plt.xlabel('Cycle Number', fontsize=12)
plt.ylabel('Discharge Capacity (Ah)', fontsize=12)
plt.title('NMC Cell State of Health (SOH): Actual vs. Predicted Degradation', fontsize=14, fontweight='bold')
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()